In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors
import numpy as np


In [ ]:

def plot_corridor_density_standalone(can_file):
    """
    Parses a streaming .CAN file, computes the integrated PAI for every (x, y) cell,
    and plots a top-down layout view without relying on py_wake_ellipsys.
    """
    outfile = 'Corridor_40mforest_512m_corr60_PAI.png'
    
    # 1. Parse the streaming .CAN structure
    print(f"Reading and parsing {can_file}...")
    with open(can_file, 'r') as f:
        header1 = f.readline().split()
        header2 = f.readline().split()
        data_tokens = f.read().split()

    nx, ny = int(header1[0]), int(header1[1])
    x_min, x_max, y_min, y_max = map(float, header2)
    data_stream = np.array(data_tokens, dtype=float)
    
    # Define vertical resolution layers based on file content structure
    nz_max = 19  
    
    # Initialize a 2D matrix to store the integrated PAI for each grid coordinate
    pai_2d = np.zeros((nx, ny))
    
    stream_ptr = 0
    for i in range(nx):
        for j in range(ny):
            h_val = data_stream[stream_ptr]
            
            if h_val <= 1.00:
                # Open corridor space (PAD is 0 everywhere)
                pai_2d[i, j] = 0.0
                stream_ptr += 2
            else:
                # Tree canopy space: extract the 19 PAD values
                profile_pads = data_stream[stream_ptr + 1 : stream_ptr + 1 + nz_max]
                
                # PAI is the vertical integral (sum) of PAD across all height layers
                pai_2d[i, j] = np.sum(profile_pads)
                
                stream_ptr += (1 + nz_max + 1)

    # 2. Reconstruct spatial 2D coordinate arrays for plotting
    x_coords = np.linspace(x_min, x_max, nx)
    y_coords = np.linspace(y_min, y_max, ny)
    X, Y = np.meshgrid(x_coords, y_coords, indexing='ij')

    # 3. Setup the custom colormap logic
    max_pai = 4.0 
    orig_cmap = plt.cm.Greens
    colormap = orig_cmap(np.linspace(0.0, 0.9, 100))
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list("forest_cmap", colormap)
    
    # 4. Generate the top-down plan view plot
    print(f"Generating PAI plot...")
    plt.figure(figsize=(8, 7))
    
    # Plot using pcolormesh with explicit vmin/vmax normalization
    im = plt.pcolormesh(X, Y, pai_2d, cmap=cmap, vmin=0, vmax=max_pai, shading='auto')
    
    # Formatting layout
    cbar = plt.colorbar(im, label=r'PAI [m$^2$/m$^2$]') # Area index standard units are m²/m²
    plt.title(f'Top-Down Canopy Density Map (Integrated PAI)')
    plt.xlabel('Distance along corridor (x) [m]')
    plt.ylabel('Lateral distance (y) [m]')
    
    # Make the grid aspect ratio square so the layout geometry isn't warped
    plt.gca().set_aspect('equal', adjustable='box')
    plt.tight_layout()
    
    # plt.savefig(outfile, dpi=300)
    # print(f"Plot saved successfully as {outfile}")
    plt.show()


In [ ]:

if __name__ == "__main__":
    # Update filename if necessary to target your file variants
    plot_corridor_density_standalone('corridor_4dx_512m.CAN')